# 26 Local Feature Engineering
Local-only deterministic feature engineering for Bluesky engagement prediction.


## 1) Load Inputs and Inspect Schemas


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

from src.features.feature_engineering import (
    FeatureConfig,
    build_local_engagement_feature_table,
    load_actor_profiles_file,
    load_hydrated_metrics_file,
    select_best_actor_profile_source_file,
    select_best_hydrated_source_file,
)
from src.source_paths import resolve_bluesky_source_root

prepared_path = Path("local/derived/bluesky/bluesky_posts_prepared.parquet")
best_matches_path = Path("local/derived/matching/bluesky_post_best_trend_matches.parquet")
full_matches_path = Path("local/derived/matching/bluesky_post_trend_matches.parquet")

prepared_df = pd.read_parquet(prepared_path)
best_matches_df = pd.read_parquet(best_matches_path)
full_matches_df = pd.read_parquet(full_matches_path)

print("prepared:", len(prepared_df), "rows", len(prepared_df.columns), "cols")
print("best matches:", len(best_matches_df), "rows", len(best_matches_df.columns), "cols")
print("full matches:", len(full_matches_df), "rows", len(full_matches_df.columns), "cols")


In [ ]:
print("Prepared columns:", prepared_df.columns.tolist())
print()
print("Best-match columns:", best_matches_df.columns.tolist())
print()
print("Full-match columns:", full_matches_df.columns.tolist())


## 2) Select Local Hydrated and Actor Sources


In [ ]:
bluesky_source_root = resolve_bluesky_source_root()
hydrated_selection = select_best_hydrated_source_file(prepared_df, base_dir=bluesky_source_root)
actor_selection = select_best_actor_profile_source_file(prepared_df, base_dir=bluesky_source_root)

hydrated_selection["selected"], actor_selection["selected"]


In [ ]:
hydrated_path = Path(hydrated_selection["selected"]["file_path"])
actor_path = Path(actor_selection["selected"]["file_path"])

hydrated_df = load_hydrated_metrics_file(hydrated_path)
actor_df = load_actor_profiles_file(actor_path)

print("selected hydrated:", hydrated_path)
print("hydrated rows:", len(hydrated_df), "unique uri:", hydrated_df["uri"].nunique())
print("selected actor:", actor_path)
print("actor rows:", len(actor_df), "unique did:", actor_df["did"].nunique())


## 3) Build Feature Table (One Row Per `uri`)


In [ ]:
config = FeatureConfig(low_quantile=0.33, high_quantile=0.66)

features_df, summary = build_local_engagement_feature_table(
    prepared_df=prepared_df,
    best_matches_df=best_matches_df,
    full_matches_df=full_matches_df,
    hydrated_df=hydrated_df,
    actor_df=actor_df,
    config=config,
)

features_df = features_df.sort_values(["uri"], kind="stable").reset_index(drop=True)

print("feature rows:", len(features_df), "feature cols:", len(features_df.columns))
summary["labeling"]


## 4) Join Coverage and Label Distribution


In [ ]:
summary["join_coverage"]


In [ ]:
print("Label distribution:")
print(pd.Series(summary["label_distribution"]))
print()
print("Match stage distribution:")
print(pd.Series(summary["match_stage_distribution"]))


## 5) Feature Groups and Leakage Guard


In [ ]:
feature_groups = summary["feature_groups"]
print("model feature count:", len(feature_groups["model_feature_columns"]))
print("label columns:", feature_groups["label_columns"])
print("debug columns count:", len(feature_groups["debug_columns"]))

leakage_columns = {"eng_like_count", "eng_reply_count", "eng_repost_count", "eng_quote_count", "engagement_total", "engagement_label"}
print()
print("leakage columns in model features:", sorted(leakage_columns.intersection(feature_groups["model_feature_columns"])))


## 6) Quality Checks and Examples


In [ ]:
missing_top = sorted(summary["missingness_by_model_feature"].items(), key=lambda kv: (-kv[1], kv[0]))[:15]
print("Top model-feature missingness:")
for key, value in missing_top:
    print(f"- {key}: {value}")


In [ ]:
features_df[[
    "uri",
    "engagement_total",
    "engagement_label",
    "match_stage",
    "has_trend_match",
    "candidate_count",
    "matched_candidate_count",
    "post_token_count",
    "post_unique_token_count",
    "actor_followers_count",
]].head(15)


## 7) Write Outputs


In [ ]:
out_dir = Path("local/derived/features")
out_dir.mkdir(parents=True, exist_ok=True)
sample_dir = Path("data/samples")
sample_dir.mkdir(parents=True, exist_ok=True)

full_out = out_dir / "bluesky_engagement_features.parquet"
sample_parquet_out = sample_dir / "bluesky_engagement_features_sample_1000.parquet"
sample_csv_out = sample_dir / "bluesky_engagement_features_sample_1000.csv"
summary_out = out_dir / "bluesky_engagement_feature_summary.json"

features_df.to_parquet(full_out, index=False)
features_df.head(1000).to_parquet(sample_parquet_out, index=False)
features_df.head(1000).to_csv(sample_csv_out, index=False)

summary_payload = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "26_local_feature_engineering",
    "input_paths": {
        "prepared_posts": str(prepared_path),
        "best_matches": str(best_matches_path),
        "full_matches": str(full_matches_path),
        "selected_hydrated_source": str(hydrated_path),
        "selected_actor_source": str(actor_path),
    },
    "hydrated_selection": hydrated_selection,
    "actor_selection": actor_selection,
    "summary": summary,
    "output_paths": {
        "full_features_parquet": str(full_out),
        "sample_features_parquet": str(sample_parquet_out),
        "sample_features_csv": str(sample_csv_out),
    },
}
summary_out.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("wrote", full_out)
print("wrote", sample_parquet_out)
print("wrote", sample_csv_out)
print("wrote", summary_out)


## 8) Readiness Statement
Phase 26 is complete when this notebook runs end-to-end, outputs are written, and summary metrics are documented in findings.
